# Steel Defect Detection Pipeline
Pipeline untuk deteksi dan segmentasi defect pada baja menggunakan deep learning.

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import os
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import segmentation_models_pytorch as smp

import warnings
warnings.filterwarnings('ignore')

In [ ]:
DATA_DIR = Path('severstal-steel-defect-detection')
TRAIN_DIR = DATA_DIR / 'train_images'
TEST_DIR = DATA_DIR / 'test_images'
TRAIN_CSV = DATA_DIR / 'train.csv'
SAMPLE_SUB = DATA_DIR / 'sample_submission.csv'

IMG_HEIGHT = 256
IMG_WIDTH = 512
NUM_CLASSES = 4
BATCH_SIZE = 4
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Using device: {DEVICE}')

In [ ]:
USE_SUBSET = False
SUBSET_SIZE = 500
USE_MIXED_PRECISION = torch.cuda.is_available()

if USE_SUBSET:
    print(f'Using subset of {SUBSET_SIZE} images for faster training')
if USE_MIXED_PRECISION:
    print('Using mixed precision training for better performance')

In [ ]:
def rle_to_mask(rle_string, height=256, width=1600):
    if pd.isna(rle_string) or rle_string == '':
        return np.zeros((height, width), dtype=np.uint8)
    
    s = rle_string.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0::2], s[1::2])]
    starts -= 1
    ends = starts + lengths
    img = np.zeros(height * width, dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape((height, width), order='F')

def mask_to_rle(mask):
    pixels = mask.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs) if len(runs) > 0 else ''

def build_masks(df, image_id, height=256, width=1600):
    masks = np.zeros((height, width, NUM_CLASSES), dtype=np.float32)
    for class_id in range(1, NUM_CLASSES + 1):
        mask_data = df[(df['ImageId'] == image_id) & (df['ClassId'] == class_id)]
        if not mask_data.empty:
            rle = mask_data.iloc[0]['EncodedPixels']
            masks[:, :, class_id - 1] = rle_to_mask(rle, height, width)
    return masks

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB)

print(f'Training data shape: {train_df.shape}')
print(f'Sample submission shape: {sample_sub.shape}')
print(f'\nFirst rows of training data:')
print(train_df.head())
print(f'\nDefect class distribution:')
print(train_df['ClassId'].value_counts().sort_index())

In [ ]:
def visualize_sample(image_id, df, img_dir):
    img_path = img_dir / image_id
    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    masks = build_masks(df, image_id, image.shape[0], image.shape[1])
    
    fig, axes = plt.subplots(1, NUM_CLASSES + 1, figsize=(20, 4))
    axes[0].imshow(image)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    colors = ['Reds', 'Greens', 'Blues', 'Purples']
    for i in range(NUM_CLASSES):
        axes[i + 1].imshow(image)
        axes[i + 1].imshow(masks[:, :, i], alpha=0.5, cmap=colors[i])
        axes[i + 1].set_title(f'Class {i + 1}')
        axes[i + 1].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
sample_images = train_df['ImageId'].unique()[:3]
for img_id in sample_images:
    visualize_sample(img_id, train_df, TRAIN_DIR)

In [ ]:
class SteelDataset(Dataset):
    def __init__(self, df, image_ids, img_dir, transform=None, is_test=False):
        self.df = df
        self.image_ids = image_ids
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_path = self.img_dir / image_id
        
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.is_test:
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            return image, image_id
        
        masks = build_masks(self.df, image_id, image.shape[0], image.shape[1])
        
        if self.transform:
            augmented = self.transform(image=image, mask=masks)
            image = augmented['image']
            masks = augmented['mask']
        
        masks = masks.permute(2, 0, 1)
        return image, masks

In [ ]:
train_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [ ]:
all_image_ids = train_df['ImageId'].unique()

if USE_SUBSET:
    all_image_ids = all_image_ids[:SUBSET_SIZE]
    print(f'Using {len(all_image_ids)} images')

train_ids, val_ids = train_test_split(all_image_ids, test_size=0.2, random_state=42)

train_dataset = SteelDataset(train_df, train_ids, TRAIN_DIR, transform=train_transform)
val_dataset = SteelDataset(train_df, val_ids, TRAIN_DIR, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Training samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')

In [ ]:
def dice_coefficient(preds, targets, smooth=1e-6):
    preds = torch.sigmoid(preds)
    preds = (preds > 0.5).float()
    
    intersection = (preds * targets).sum(dim=(2, 3))
    union = preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return dice.mean()

class DiceLoss(nn.Module):
    def __init__(self):
        super(DiceLoss, self).__init__()
    
    def forward(self, preds, targets, smooth=1e-6):
        preds = torch.sigmoid(preds)
        
        intersection = (preds * targets).sum(dim=(2, 3))
        union = preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        
        dice = (2.0 * intersection + smooth) / (union + smooth)
        return 1 - dice.mean()

In [ ]:
model = smp.Unet(
    encoder_name='mobilenet_v2',
    encoder_weights='imagenet',
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None
)

model = model.to(DEVICE)
print(f'Model loaded on {DEVICE}')

In [ ]:
criterion = DiceLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
scaler = GradScaler() if USE_MIXED_PRECISION else None

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, scaler=None):
    model.train()
    total_loss = 0
    total_dice = 0
    
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        if scaler is not None:
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
        
        total_loss += loss.item()
        total_dice += dice_coefficient(outputs, masks).item()
    
    return total_loss / len(loader), total_dice / len(loader)

def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    total_dice = 0
    
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            total_loss += loss.item()
            total_dice += dice_coefficient(outputs, masks).item()
    
    return total_loss / len(loader), total_dice / len(loader)

In [ ]:
best_dice = 0
history = {'train_loss': [], 'train_dice': [], 'val_loss': [], 'val_dice': []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_dice = train_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler)
    val_loss, val_dice = validate_epoch(model, val_loader, criterion, DEVICE)
    
    history['train_loss'].append(train_loss)
    history['train_dice'].append(train_dice)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_dice)
    
    scheduler.step(val_loss)
    
    print(f'Epoch {epoch+1}/{NUM_EPOCHS}')
    print(f'Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f}')
    print(f'Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}')
    
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'Model saved with Dice: {best_dice:.4f}')
    print('-' * 50)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()

axes[1].plot(history['train_dice'], label='Train Dice')
axes[1].plot(history['val_dice'], label='Val Dice')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice Coefficient')
axes[1].set_title('Training and Validation Dice')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
def predict_image(model, image, device):
    model.eval()
    with torch.no_grad():
        image = image.unsqueeze(0).to(device)
        output = model(image)
        output = torch.sigmoid(output)
        output = (output > 0.5).float()
    return output.squeeze(0).cpu().numpy()

def visualize_prediction(model, image_id, df, img_dir, device):
    img_path = img_dir / image_id
    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    original_shape = image.shape[:2]
    
    augmented = val_transform(image=image)
    image_tensor = augmented['image']
    
    pred_masks = predict_image(model, image_tensor, device)
    
    fig, axes = plt.subplots(2, NUM_CLASSES + 1, figsize=(20, 8))
    
    axes[0, 0].imshow(image)
    axes[0, 0].set_title('Original Image')
    axes[0, 0].axis('off')
    axes[1, 0].axis('off')
    
    colors = ['Reds', 'Greens', 'Blues', 'Purples']
    
    gt_masks = build_masks(df, image_id, original_shape[0], original_shape[1])
    
    for i in range(NUM_CLASSES):
        axes[0, i + 1].imshow(image)
        gt_resized = cv2.resize(gt_masks[:, :, i], (IMG_WIDTH, IMG_HEIGHT))
        axes[0, i + 1].imshow(gt_resized, alpha=0.5, cmap=colors[i])
        axes[0, i + 1].set_title(f'GT Class {i + 1}')
        axes[0, i + 1].axis('off')
        
        axes[1, i + 1].imshow(image)
        axes[1, i + 1].imshow(pred_masks[i], alpha=0.5, cmap=colors[i])
        axes[1, i + 1].set_title(f'Pred Class {i + 1}')
        axes[1, i + 1].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))

for img_id in val_ids[:3]:
    visualize_prediction(model, img_id, train_df, TRAIN_DIR, DEVICE)

In [ ]:
test_images = sorted([f.name for f in TEST_DIR.glob('*.jpg')])
test_dataset = SteelDataset(None, test_images, TEST_DIR, transform=val_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model.load_state_dict(torch.load('best_model.pth'))
model.eval()

predictions = []

with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        outputs = torch.sigmoid(outputs)
        outputs = (outputs > 0.5).float()
        
        outputs = outputs.cpu().numpy()
        
        for i, image_id in enumerate(image_ids):
            for class_id in range(1, NUM_CLASSES + 1):
                mask = outputs[i, class_id - 1]
                mask_resized = cv2.resize(mask, (1600, 256), interpolation=cv2.INTER_NEAREST)
                rle = mask_to_rle(mask_resized)
                
                if rle == '':
                    rle = '1 409600'
                
                predictions.append({
                    'ImageId': image_id,
                    'EncodedPixels': rle,
                    'ClassId': class_id
                })

submission_df = pd.DataFrame(predictions)
submission_df.to_csv('submission.csv', index=False)
print(f'Submission file created with {len(submission_df)} rows')

## Optimisasi untuk Komputasi Ringan

**Perubahan yang dilakukan:**

1. **Ukuran Gambar**: Dikurangi dari 256x1600 menjadi **256x512** (pengurangan ~68% memory)
2. **Model**: Diganti dari ResNet34 ke **MobileNetV2** (model yang lebih ringan dan cepat)
3. **Batch Size**: Dikurangi dari 8 menjadi **4** (mengurangi penggunaan memory)
4. **Epochs**: Dikurangi dari 20 menjadi **10** epochs
5. **Learning Rate**: Dinaikkan dari 1e-4 ke **1e-3** untuk konvergensi lebih cepat
6. **Workers**: Dikurangi dari 4 menjadi **2** untuk mengurangi overhead
7. **Mixed Precision Training**: Ditambahkan untuk GPU yang support (lebih cepat dan hemat memory)
8. **Subset Training**: Opsi untuk training dengan subset data (aktifkan USE_SUBSET=True)

**Estimasi Pengurangan Komputasi:**
- Memory usage: ~70-75% lebih rendah
- Training time: ~60-70% lebih cepat per epoch
- Model size: ~50% lebih kecil

**Trade-off:**
- Akurasi mungkin sedikit lebih rendah karena resolusi lebih kecil
- Tapi masih cukup baik untuk deteksi defect steel